In [0]:
%sql
CREATE CATALOG IF NOT EXISTS demoworkspacejoby;

CREATE SCHEMA IF NOT EXISTS demoworkspacejoby.stock_bronze;
CREATE SCHEMA IF NOT EXISTS demoworkspacejoby.stock_silver;
CREATE SCHEMA IF NOT EXISTS demoworkspacejoby.stock_gold;

CREATE VOLUME IF NOT EXISTS demoworkspacejoby.stock_bronze.stock_data_staging;

In [0]:
import requests
import json
import time
from datetime import datetime, timezone

SYMBOLS = ["AAPL", "MSFT", "GOOGL", "AMZN"]

BASE_URL = "https://www.alphavantage.co/query"
API_KEY = "your_alpha_vantage_api_key_here"

VOLUME_BASE = "/Volumes/demoworkspacejoby/stock_bronze/stock_data_staging"

In [0]:
def fetch_alpha_vantage(function_name: str, symbol: str):
    params = {
        "function": function_name,
        "symbol": symbol,
        "apikey": API_KEY
    }

    if function_name == "TIME_SERIES_DAILY":
        params["outputsize"] = "compact"

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    if "Error Message" in data:
        raise Exception(f"API error for {symbol}, {function_name}: {data}")

    if "Information" in data:
        raise Exception(f"API information message for {symbol}, {function_name}: {data}")

    if "Note" in data:
        raise Exception(f"API rate limit for {symbol}, {function_name}: {data}")

    return data

In [0]:
run_ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

results = []

for symbol in SYMBOLS:
    print("=" * 70)
    print(f"Processing {symbol}")
    print("=" * 70)

    try:
        daily_data = fetch_alpha_vantage("TIME_SERIES_DAILY", symbol)
        time.sleep(15)

        quote_data = fetch_alpha_vantage("GLOBAL_QUOTE", symbol)
        time.sleep(15)

        company_data = fetch_alpha_vantage("OVERVIEW", symbol)
        time.sleep(15)

        daily_path = f"{VOLUME_BASE}/daily_prices/symbol={symbol}/daily_{run_ts}.json"
        quote_path = f"{VOLUME_BASE}/quotes/symbol={symbol}/quote_{run_ts}.json"
        company_path = f"{VOLUME_BASE}/company_info/symbol={symbol}/company_{run_ts}.json"

        dbutils.fs.put(daily_path, json.dumps(daily_data), overwrite=True)
        dbutils.fs.put(quote_path, json.dumps(quote_data), overwrite=True)
        dbutils.fs.put(company_path, json.dumps(company_data), overwrite=True)

        results.append({
            "symbol": symbol,
            "daily_path": daily_path,
            "quote_path": quote_path,
            "company_path": company_path
        })

        print(f"Saved daily data: {daily_path}")
        print(f"Saved quote data: {quote_path}")
        print(f"Saved company info: {company_path}")

    except Exception as e:
        print(f"Failed for {symbol}: {e}")

    time.sleep(15)

print("=" * 70)
print("INGESTION COMPLETE")
print("=" * 70)

for r in results:
    print(r)

Processing AAPL
Wrote 13527 bytes.
Wrote 293 bytes.
Wrote 2079 bytes.
Saved daily data: /Volumes/demoworkspacejoby/stock_bronze/stock_data_staging/daily_prices/symbol=AAPL/daily_20260429_181513.json
Saved quote data: /Volumes/demoworkspacejoby/stock_bronze/stock_data_staging/quotes/symbol=AAPL/quote_20260429_181513.json
Saved company info: /Volumes/demoworkspacejoby/stock_bronze/stock_data_staging/company_info/symbol=AAPL/company_20260429_181513.json
Processing MSFT
Wrote 13525 bytes.
Wrote 293 bytes.
Wrote 2351 bytes.
Saved daily data: /Volumes/demoworkspacejoby/stock_bronze/stock_data_staging/daily_prices/symbol=MSFT/daily_20260429_181513.json
Saved quote data: /Volumes/demoworkspacejoby/stock_bronze/stock_data_staging/quotes/symbol=MSFT/quote_20260429_181513.json
Saved company info: /Volumes/demoworkspacejoby/stock_bronze/stock_data_staging/company_info/symbol=MSFT/company_20260429_181513.json
Processing GOOGL
Wrote 13527 bytes.
Wrote 296 bytes.
Wrote 2066 bytes.
Saved daily data: /